All this file is generated by AI. It's just for analysis.

# Server vs. single-mode timings

Compare the first response's `timing.clock` value (milliseconds) for each matching sample ID across the three cache modes. Samples are sorted by `cmi-cache-only - no-cache`, largest difference first. A separate table shows per-mode percentiles using linear interpolation.

In [4]:
import json
from pathlib import Path

import pandas as pd

types_dir = Path("ocaml-lsp-17-10-2025/types")
files = {
    "no-cache": types_dir / "no-cache/main/full_responses.json",
    "cmi-cache-only": types_dir / "cmi-cache-only/main/full_responses.json",
    "full-cache": types_dir / "full-cache/main/full_responses.json",
}

def read_clock_timings(path):
    with path.open() as source:
        rows = [json.loads(line) for line in source]
    return {
        row["sample_id"]: row["responses"][0]["timing"]["clock"]
        for row in rows
    }

timings = {mode: read_clock_timings(path) for mode, path in files.items()}
sample_ids = sorted(timings["no-cache"])

assert len(sample_ids) == 91
assert all(set(mode_timings) == set(sample_ids) for mode_timings in timings.values())

table = pd.DataFrame(
    {mode: [mode_timings[sample_id] for sample_id in sample_ids]
     for mode, mode_timings in timings.items()},
    index=sample_ids,
)
table.index.name = "sample_id"
difference = table["cmi-cache-only"] - table["no-cache"]
table = table.loc[difference.sort_values(ascending=False, kind="stable").index]
with pd.option_context("display.max_rows", None):
    display(table)

percentiles = table.quantile([0.10, 0.50, 0.90, 0.95])
percentiles.index = pd.Index(["p10", "p50", "p90", "p95"], name="percentile")
display(percentiles)

,no-cache,cmi-cache-only,full-cache
sample_id,,,
87,1480,2043,447
79,1483,1932,507
90,1476,1911,356
84,1501,1923,355
69,1468,1862,1405
81,1463,1838,499
80,1447,1810,393
74,1451,1808,400
76,1490,1845,375


,no-cache,cmi-cache-only,full-cache
percentile,,,
p10,1411.0,1373.0,344.0
p50,1438.0,1477.0,427.0
p90,1485.0,1795.0,1061.0
p95,1509.0,1853.5,1254.0


## Full-cache timings and typer cache

The typer cache is recorded either as `miss` or as counts of `reused` and `typed` items. Counts are unavailable for a `miss`.

In [5]:
import shlex

with files["full-cache"].open() as source:
    full_cache_rows = [json.loads(line) for line in source]

records = []
for row in full_cache_rows:
    response = row["responses"][0]
    typer = response["cache"]["typer"]
    if typer == "miss":
        status, reused, typed = "miss", None, None
    elif isinstance(typer, dict):
        status, reused, typed = "cached", typer["reused"], typer["typed"]
    else:
        raise ValueError(f"Unexpected typer cache value: {typer!r}")
    command = shlex.split(row["cmd"])
    query = command[command.index("server") + 1]
    records.append({
        "sample_id": row["sample_id"],
        "query": query,
        "timing.clock": response["timing"]["clock"],
        "timing.typer": response["timing"]["typer"],
        "timing.query": response["timing"]["query"],
        "typer_cache": status,
        "typer_reused": reused,
        "typer_typed": typed,
    })

full_cache_table = pd.DataFrame(records).set_index("sample_id")
full_cache_table[["typer_reused", "typer_typed"]] = full_cache_table[["typer_reused", "typer_typed"]].astype("Int64")
full_cache_table = full_cache_table.sort_values("timing.clock", ascending=False, kind="stable")
with pd.option_context("display.max_rows", None):
    display(full_cache_table)

,query,timing.clock,typer_cache,typer_reused,typer_typed
sample_id,,,,,
0,case-analysis,1418,miss,<NA>,<NA>
69,expand-prefix,1405,miss,<NA>,<NA>
54,complete-prefix,1392,miss,<NA>,<NA>
70,expand-prefix,1380,miss,<NA>,<NA>
55,complete-prefix,1376,miss,<NA>,<NA>
64,expand-prefix,1132,miss,<NA>,<NA>
65,expand-prefix,1116,miss,<NA>,<NA>
49,complete-prefix,1097,miss,<NA>,<NA>
50,complete-prefix,1085,miss,<NA>,<NA>
